# BRFSS research optimization on Colab
Use the original XPT in your Drive. This notebook applies the delivered patch to the fork, so it does not require a push. Keep the final test locked during development. Results describe stroke history, not future risk. Full data stay out of git.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Upload the delivered `research-baseline.patch` (small text patch, not the dataset).

In [ ]:
from google.colab import files
uploaded = files.upload()
assert "research-baseline.patch" in uploaded


In [ ]:
import subprocess
from pathlib import Path
project = Path('/content/BRFSS-Stroke-Neural-Feature-Learning')
subprocess.run(['git','clone','https://github.com/DenisParker7/BRFSS-Stroke-Neural-Feature-Learning.git',str(project)],check=True)
subprocess.run(['git','checkout','095ae4d367d1c34000e1c235e9f4a93596c99791'],cwd=project,check=True)
subprocess.run(['git','switch','-c','research/colab-optimization'],cwd=project,check=True)
subprocess.run(['git','apply','/content/research-baseline.patch'],cwd=project,check=True)


In [ ]:
import sys
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-research.txt'],cwd=project,check=True)


In [ ]:
data_path = Path('/content/drive/MyDrive/LLCP2024.XPT')  # Set the actual path.
assert data_path.is_file(), 'Set data_path to your existing XPT file.'


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=project,check=True)


In [ ]:
run_dir = Path('/content/drive/MyDrive/StrokeResearch/development-seed42')
subprocess.run([sys.executable,'-m','truststroke.optimize','--data',str(data_path),
                '--output',str(run_dir),'--seed','42','--epochs','40','--threads','2'],cwd=project,check=True)


Optional exploratory blend search, before opening the final test. This adds three blend candidates to the search and must be reported as additional tuning.


In [ ]:
subprocess.run([sys.executable,'-m','truststroke.ensemble_search','--data',str(data_path),'--run',str(run_dir)],cwd=project,check=True)


Inspect development.json and freeze the protocol before final testing. Run the following cell only once; do not tune based on its results. The command refuses to overwrite an existing locked-test result.

In [ ]:
subprocess.run([sys.executable,'-m','truststroke.evaluate_locked','--data',str(data_path),
                '--run',str(run_dir),'--bootstrap','500'],cwd=project,check=True)
